# Ingestion Pipeline Runner

Interactive notebook to run each ingestion stage with tqdm progress bars.
One cell per stage — run them in order.

In [ ]:
# ── Cell 0: Setup ──────────────────────────────────────────────────────────
import json, sys, time, logging
from pathlib import Path

import yaml
from tqdm.notebook import tqdm

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

logging.basicConfig(level=logging.WARNING)
logging.getLogger("sqlalchemy.engine").setLevel(logging.WARNING)
logging.getLogger("LiteLLM").setLevel(logging.WARNING)
logging.getLogger("httpx").setLevel(logging.WARNING)

from db.connection import get_session
from db.models import SourceObservation, ReconciledSpecies
from ingestion.sources import firstnature, funghiitaliani, mushroomexpert, ultimatemushroom, wikipedia

FETCH_SOURCES = [wikipedia, firstnature, mushroomexpert, ultimatemushroom, funghiitaliani]

with open(ROOT / "data" / "seed" / "species_list.yaml") as f:
    species_list = yaml.safe_load(f).get("species", [])

print(f"{len(species_list)} species, {len(FETCH_SOURCES)} sources")

In [ ]:
# ── Cell 1: Current status ────────────────────────────────────────────────
from sqlalchemy import func, distinct, or_

s = get_session()
l1 = s.query(SourceObservation).count()
l1_sp = s.query(func.count(distinct(SourceObservation.scientific_name))).scalar()
l2 = s.query(ReconciledSpecies).count()
l3 = s.query(ReconciledSpecies).filter(ReconciledSpecies.embedding_morphological != None).count()
s.close()

total_expected = len(species_list) * len(FETCH_SOURCES)
print(f"Layer 1: {l1}/{total_expected} observations ({l1_sp} species)")
print(f"Layer 2: {l2}/{len(species_list)} reconciled")
print(f"Layer 3: {l3}/{l2} embedded")

In [ ]:
# ── Cell 2: Fetch (Stage 2) ───────────────────────────────────────────────
ok, failed = 0, []

pbar = tqdm(total=len(species_list) * len(FETCH_SOURCES), desc="Fetching", unit="page")
for entry in species_list:
    name = entry["scientific_name"]
    aliases = entry.get("aliases", [])
    for source in FETCH_SOURCES:
        result = source.fetch_species_page(name, aliases=aliases)
        if result:
            ok += 1
        else:
            failed.append(f"{source.SOURCE_NAME}/{name}")
        pbar.set_postfix(ok=ok, fail=len(failed), species=name[:20])
        pbar.update(1)
pbar.close()

print(f"\nDone: {ok} fetched, {len(failed)} missed")
if failed:
    print(f"Misses: {', '.join(failed[:10])}" + (f" +{len(failed)-10} more" if len(failed) > 10 else ""))

In [ ]:
# ── Cell 3: Extract (Stage 3) ─────────────────────────────────────────────
from ingestion.extract import extract_features_from_text, save_extraction

session = get_session()
already_done = {
    (row[0], row[1])
    for row in session.query(
        SourceObservation.scientific_name, SourceObservation.source_name
    ).all()
}
session.close()

# Build work list (skip already-extracted)
work = []
for entry in species_list:
    name = entry["scientific_name"]
    aliases = entry.get("aliases", [])
    for source in FETCH_SOURCES:
        if (name, source.SOURCE_NAME) not in already_done:
            work.append((name, aliases, source))

print(f"{len(already_done)} already done, {len(work)} to extract")

ok, failed = 0, []
pbar = tqdm(work, desc="Extracting", unit="obs")
for name, aliases, source in pbar:
    pbar.set_postfix(species=name[:20], src=source.SOURCE_NAME[:12])
    page = source.fetch_species_page(name, aliases=aliases)
    if page is None:
        continue
    try:
        features = extract_features_from_text(name, page["text"], source_name=source.SOURCE_NAME)
        save_extraction(features, page["url"], page["text"], source.SOURCE_NAME)
        ok += 1
    except Exception as e:
        failed.append(f"{source.SOURCE_NAME}/{name}")
        pbar.write(f"FAIL: {source.SOURCE_NAME}/{name}: {e}")
pbar.close()

print(f"\nDone: {ok} extracted, {len(failed)} failed")
if failed:
    print(f"Failed: {', '.join(failed)}")

In [ ]:
# ── Cell 4: Reconcile (Stage 4) ───────────────────────────────────────────
from ingestion.reconcile import reconcile_species

session = get_session()
names = [
    row[0]
    for row in session.query(SourceObservation.scientific_name).distinct().all()
]
session.close()

print(f"{len(names)} species to reconcile")

ok, skipped, failed = 0, 0, []
pbar = tqdm(names, desc="Reconciling", unit="sp")
for name in pbar:
    pbar.set_postfix(species=name[:25])
    s = get_session()
    try:
        result = reconcile_species(s, name)
        if result:
            ok += 1
        else:
            skipped += 1
    except Exception as e:
        failed.append(name)
        pbar.write(f"FAIL: {name}: {e}")
    finally:
        s.close()
pbar.close()

print(f"\nDone: {ok} reconciled, {skipped} skipped, {len(failed)} failed")
if failed:
    print(f"Failed: {', '.join(failed)}")

In [ ]:
# ── Cell 5: Embed (Stage 5) ───────────────────────────────────────────────
from sqlalchemy import or_
from ingestion.embed import embed_species

session = get_session()
pending = (
    session.query(ReconciledSpecies)
    .filter(
        or_(
            ReconciledSpecies.embedded_at.is_(None),
            ReconciledSpecies.embedded_at < ReconciledSpecies.reconciled_at,
        )
    )
    .all()
)

print(f"{len(pending)} species to embed")

ok, failed = 0, []
pbar = tqdm(pending, desc="Embedding", unit="sp")
for species in pbar:
    pbar.set_postfix(species=species.scientific_name[:25])
    try:
        embed_species(session, species)
        ok += 1
    except Exception as e:
        session.rollback()
        failed.append(species.scientific_name)
        pbar.write(f"FAIL: {species.scientific_name}: {e}")
pbar.close()
session.close()

print(f"\nDone: {ok} embedded, {len(failed)} failed")
if failed:
    print(f"Failed: {', '.join(failed)}")

In [ ]:
# ── Cell 6: Final status ──────────────────────────────────────────────────
from sqlalchemy import func, distinct

s = get_session()
l1 = s.query(SourceObservation).count()
l1_sp = s.query(func.count(distinct(SourceObservation.scientific_name))).scalar()
l2 = s.query(ReconciledSpecies).count()
l3 = s.query(ReconciledSpecies).filter(ReconciledSpecies.embedding_morphological != None).count()
review = s.query(ReconciledSpecies).filter(ReconciledSpecies.needs_review == True).count()

l1_names = {r[0] for r in s.query(distinct(SourceObservation.scientific_name)).all()}
l2_names = {r[0] for r in s.query(ReconciledSpecies.scientific_name).all()}
missing = l1_names - l2_names
s.close()

print(f"Layer 1: {l1} observations ({l1_sp} species)")
print(f"Layer 2: {l2} reconciled")
print(f"Layer 3: {l3} embedded")
print(f"Needs review: {review}")
if missing:
    print(f"Missing from L2: {', '.join(sorted(missing))}")
else:
    print("All species reconciled + embedded")